# Гир 2 v0 — мультимонетный симулятор фиксированной модели 1.0

Трек **модели (M)**, не живой бот. **Статус:** гир 2 = закрыт (контур; 2.2 вне scope). Этот ноутбук — рабочая копия с узким debug `CONFIG`; **канон счётчиков** — [`docs/gear-2-close-20260825.md`](docs/gear-2-close-20260825.md), не executed cells.

**Выход прогона:** таблица сделок (сигнал vs fill), без графиков.

**Контур v0**

- Вселенная: все **crypto** USDT-perp (`research.is_crypto`), **без акций**.
- Данные: компактные lean-тики `output/lean_ticks` (полное L1). Строки со stale-cross (skew/age > 2 с, как в сборщике) из набора убраны; дыры дат остаются честными.
- На каждом тике — та же elif-цепочка, что в [`model.ipynb`](model.ipynb) (гир 1.0): open long → open short → close long → close short + гейты A/B/объём. Движок: `research/gear2_backtest.py`.
- Слот **K=1**: после входа можно только выйти из **этой** монеты. Пока `pending`, чужие монеты с порогом open считаются в `n_filtered_pending_skip` (не в `slot_busy`).
- MA / окно усреднения — **на монету**. Fill — первый тик **той же** монеты с `ts ≥ signal + Trade_Lat`.
- Этап 3: **A** = `USE_REGIME_TOPN=False` (эталон). **B** = Top‑10 только на **open**. C (`regime_on`) и D (случайный вход) — не здесь (гир 2.2).
- `Trade_Lat` / `fee_rate` / правило `max_latency_*` — заморозка гира 1.0; не подбирать.

**Не входит:** поиск порогов (гир 3), политика размера (гир 2.5), живые заявки, прибыльность на коротком ряде, C/D.

Лестница: [`docs/strategy-gears.md`](docs/strategy-gears.md). Снимок закрытия: [`docs/gear-2-close-20260825.md`](docs/gear-2-close-20260825.md).

# Покрытие локальных данных для гира 2

Снимок **Mac**, каталог репозитория. Время везде **UTC**.  
Дата описи: 2026-08-19.

Тики: `output/lean_ticks/` — компактные 5-минутные parquet, lean 16 колонок.  
Бары: `output/okx_bar5m_hist_regime/`, `output/bybit_bar5m_hist_regime/` — hist REST, не live-слой сборщика.

Ноутбук: [`model_gear2.ipynb`](../model_gear2.ipynb), `LEAN_TICKS = output/lean_ticks`.  
Пути и схема: [`model-data-sources.md`](model-data-sources.md).

Дыры дат и часов — честные: в эти минуты тиков нет. Не считать тишиной рынка.

Строки, где одна нога книги старше 2 с (ложный спред), из тиков **уже вырезаны** (правило сборщика fail-closed). Generation-suppress в parquet нет.

---

## 1. Тики — сводка

| | |
|--|--|
| Файлов | 4495 окон по 5 минут |
| Дни с тиками | 21 календарный день: **2026-07-22 … 2026-08-19** |
| Сплошных суток (288/288) | 6, 7, 8, 9, 11, 12, 13, 15 августа |
| Вселенная | все монеты окна; гир 2 режет crypto через `is_crypto` |

**Нет тиков совсем:** 2026-07-26 … 2026-08-02 (включительно).  
Также нет начала 22.07 (до 11:05) и продолжения 19.08 после 12:00.

### Сколько 5-минутных окон в сутках

Полные сутки = 288 окон. Час = 12 окон.

| День UTC | Окон | Доля | Смысл покрытия |
|----------|------|------|----------------|
| 2026-07-22 | 51 | 18% | хвост дня: 11:05–12:15 и 20:55–24:00 |
| 2026-07-23 | 245 | 85% | с полуночи до 20:25 |
| 2026-07-24 | 158 | 55% | 07:50–16:55 и 19:55–24:00 |
| 2026-07-25 | 19 | 7% | два коротких куска: ~00:45–01:35 и 10:40–11:25 |
| 2026-07-26 … 08-02 | 0 | — | **дыра** |
| 2026-08-03 | 125 | 43% | с 13:35 до конца суток |
| 2026-08-04 | 141 | 49% | полночь–11:45 |
| 2026-08-05 | 146 | 51% | с 11:50 до конца суток |
| 2026-08-06 | 288 | 100% | сутки целиком |
| 2026-08-07 | 288 | 100% | сутки целиком |
| 2026-08-08 | 288 | 100% | сутки целиком |
| 2026-08-09 | 288 | 100% | сутки целиком |
| 2026-08-10 | 132 | 46% | куски: ночь, день, вечер (см. §2) |
| 2026-08-11 | 288 | 100% | сутки целиком |
| 2026-08-12 | 288 | 100% | сутки целиком |
| 2026-08-13 | 288 | 100% | сутки целиком |
| 2026-08-14 | 287 | 99.7% | сутки минус одно окно 12:25–12:30 |
| 2026-08-15 | 288 | 100% | сутки целиком |
| 2026-08-16 | 253 | 88% | почти все часы, внутри часа дыры по 5 мин |
| 2026-08-17 | 247 | 86% | то же |
| 2026-08-18 | 267 | 93% | день почти сплошной; ночь/вечер — редкие 5-мин дыры |
| 2026-08-19 | 120 | 42% | только 00:00–12:00, с редкими 5-мин дырами |

---

## 2. Тики — часы UTC

Сетка: каждая клетка — один час.  
`.` = нет окон; число = сколько из 12 пятиминуток есть; `##` = час полный (12/12).

```text
час   00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
07-22  .  .  .  .  .  .  .  .  .  .  . 11  3  .  .  .  .  .  .  .  1 ## ## ##
07-23 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##  5  .  .  .
07-24  .  .  .  .  .  .  .  2 ## ## ## ## ## ## ## ## 11  .  .  1 ## ## ## ##
07-25  3  7  .  .  .  .  .  .  .  .  4  5  .  .  .  .  .  .  .  .  .  .  .  .
08-03  .  .  .  .  .  .  .  .  .  .  .  .  .  5 ## ## ## ## ## ## ## ## ## ##
08-04 ## ## ## ## ## ## ## ## ## ## ##  9  .  .  .  .  .  .  .  .  .  .  .  .
08-05  .  .  .  .  .  .  .  .  .  .  .  2 ## ## ## ## ## ## ## ## ## ## ## ##
08-06 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-07 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-08 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-09 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-10 ## ## ## ## ##  7  .  .  .  .  .  .  1  9  .  .  7  1  .  7  4 ## ## ##
08-11 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-12 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-13 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-14 ## ## ## ## ## ## ## ## ## ## ## ## 11 ## ## ## ## ## ## ## ## ## ## ##
08-15 ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ## ##
08-16 ## ## ## ## ## 11 11 10 10 10 11 10 10 10 10 11 10  8 10 10 10 11 10 10
08-17 11 10 10 10 10 11 10 10 10 10 11 10 11 10 10 10 11 10 11 10 10 11 10 10
08-18 11 10 10 11 10 10 11 ## ## ## ## ## ## ## ## ## ## ## 11 10 10 10 10 11
08-19 10 10 10 10 10 10 10 10 10 10 10 10  .  .  .  .  .  .  .  .  .  .  .  .
```

### Непрерывные куски (начало включительно, конец исключительно)

Конец `00:00` = до полуночи, последнее окно начинается в 23:55.

| День | Интервалы UTC |
|------|----------------|
| 22.07 | 11:05–12:15; 20:55–24:00 |
| 23.07 | 00:00–20:25 |
| 24.07 | 07:50–16:55; 19:55–24:00 |
| 25.07 | 00:45–01:35; 10:40–11:25 |
| 03.08 | 13:35–24:00 |
| 04.08 | 00:00–11:45 |
| 05.08 | 11:50–24:00 |
| 06–09.08 | 00:00–24:00 |
| 10.08 | 00:00–05:35; 12:55–13:45; 16:25–17:05; 19:00–19:35; 20:40–24:00 |
| 11–13.08 | 00:00–24:00 |
| 14.08 | 00:00–12:25 и 12:30–24:00 (нет только 12:25–12:30) |
| 15.08 | 00:00–24:00 |
| 16–17.08 | все часы есть, но внутри часа не хватает 1–4 окон по 5 мин |
| 18.08 | ночь с редкими 5-мин дырами; **06:15–18:50 почти сплошь**; вечер снова с дырами |
| 19.08 | 00:00–12:00 с редкими 5-мин дырами; после полудня тиков нет |

16–19 августа: это не «пустые часы», а пропуски отдельных 5-минутных файлов (часто 10–11 из 12 окон в часе).

---

## 3. Бары `5m` (hist)

Нужны скринеру гира 1.5, не контуру v0 гира 2. Календарь **шире** тиков: warmup режима можно считать там, где тиков ещё нет.

| | OKX | Bybit |
|--|-----|-------|
| Путь | `output/okx_bar5m_hist_regime/` | `output/bybit_bar5m_hist_regime/` |
| Монет | 336 (crypto ≈ 240) | 336 (crypto ≈ 240) |
| Дни | **2026-07-08 … 2026-08-19** | то же |
| Дыр в календаре | нет | нет |
| Часовая нарезка | сутки целиком (REST hist) | то же |

Дни, где есть тики, барами закрыты. В hist-барах нет **QNT** и **USDC** (эти тикеры есть в тиках).

---

## 4. Как выбирать окно в ноутбуке

`START` / `END` — UTC ISO, `END` не входит в срез.

| Задача | Окно |
|--------|------|
| Сутки без дыр | любой из 6, 7, 8, 9, 11, 12, 13, 15 августа |
| Почти сутки, одна 5-мин дыра | 14 августа |
| Длинный сплошной кусок днём | 18 августа 06:15–18:50 |
| Максимум истории | 22.07–19.08, помня июльскую дыру и куски часов |

Пример полных суток:

```text
START = "2026-08-11T00:00:00Z"
END   = "2026-08-12T00:00:00Z"
```


In [42]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

REPO = Path(".").resolve()
if not (REPO / "research" / "is_crypto.py").exists():
    REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))

from IPython.display import display

from research.is_crypto import is_crypto
from research.lean_ticks_io import parse_ts_ms, read_and_prepare_lean_ticks, gear2_lean_columns
from research.gear2_regime_topn import (
    OKX_BAR_ROOT,
    build_topn_by_bar,
    canon_ma_params,
    completed_bar_start_ms,
    describe_canon,
    load_crypto_feature_frames,
    no_hist_tick_coins,
    preview_topn_at,
)

## CONFIG

Узкое окно и `MAX_COINS` — для отладки цикла. Полный dump: `MAX_COINS = 0` и расширьте `START`/`END`.

**Этот CONFIG — debug**, не канон 4h: `END` 02:00 UTC. Канон счётчиков A/B — [`docs/gear-2-close-20260825.md`](docs/gear-2-close-20260825.md) (`2026-08-18 00:00–04:00 UTC`, все crypto). Saved outputs зачищены 2026-08-25, чтобы файл нельзя было читать как воспроизведение таблицы.

### День без OOM (RAM)

Полный день ≈ 60–100M тиков / DF ~19 GB / пик ~50–65 GB — на типичном ноутбуке (16–32 GB) **монолитный** прогон убивает kernel. Путь:

1. **`DAY_MODE=True`** → `run_backtest_chunked` (чанки `CHUNK_HOURS`, carry K=1 + warmup MA). Пик ≈ один чанк (~2 ч полный crypto ≈ несколько GB после prune), не весь день.
2. **`PRUNE_LEAN_COLS=True`** + `coins=` в reader (не грузить U целиком) + `RUN_ARM_B=False` на день (только arm A).
3. Опционально **`MAX_COINS` / `COIN_ALLOWLIST`** — честный «день на подмножестве», не full-universe.

Default ниже оставляет 2h debug. Для 1 календарного дня: `DAY_MODE=True`, `END` = следующий UTC midnight, `RUN_ARM_B=False`.

Заморозка гира 1.0 (не ретьюнить): `Trade_Lat=100`, `fee_rate=0.00075`. `max_latency_*` после загрузки = pooled p95 среза (в DAY_MODE — p95 первого чанка).

`USE_REGIME_TOPN=False` — **arm A** (эталон v0). C/D не реализованы.


In [ ]:
# --- данные ---
LEAN_TICKS = REPO / "output" / "lean_ticks"

# UTC ISO. END exclusive.
# Close-out A/B (all crypto 4h): docs/gear-2-close-20260825.md
# 2026-08-18T00:00:00Z → 2026-08-18T04:00:00Z, MAX_COINS=0
# 1 calendar day (chunked): START day 00:00Z, END next 00:00Z, DAY_MODE=True
START = "2026-08-16T00:00:00Z"
END = "2026-08-18T02:00:00Z"

MAX_COINS = 15  # 0 = все crypto в срезе; >0 = алфавитный срез для отладки
COIN_ALLOWLIST = ["BTC", "ETH", "SOL", "XRP", "ADA"]  # list[str], например ["BTC", "ETH"]; иначе is_crypto
K = 1  # v0: один слот
assert K == 1, "gear 2 v0 only supports K=1"

# --- RAM / day path (defaults keep 2h debug intact) ---
DAY_MODE = True          # True → run_backtest_chunked; peak ≈ one chunk
CHUNK_HOURS = 2           # 1–2 recommended on 16GB laptop for full crypto
PRUNE_LEAN_COLS = True    # drop sizes/trigger at Arrow read when Check_volume=False
RUN_ARM_B = True          # False on full day: skip dual A+B (arm A only)
# Expected peak (order of magnitude, Check_volume=False, prune on):
#   2h full crypto monolithic ~4–8 GB; 24h monolithic ~50–65 GB (OOM);
#   DAY_MODE chunk 2h ~4–8 GB peak; subset MAX_COINS=20 day ≪ 2 GB.

# Gear 1.5 arm B (Top-N cluster) — ENTRY only, not Gate B / not close.
USE_REGIME_TOPN = False  # arm A baseline; B runs in the close-out table when RUN_ARM_B
REGIME_TOP_N = 10
REGIME_BAR_ROOT = OKX_BAR_ROOT  # output/okx_bar5m_hist_regime
REGIME_TOPN_WORKERS = 8

VARIATION = {
    "thresh_open_long": 0.02,
    "thresh_open_short": 0.02,
    "thresh_close_long": 0.00,
    "thresh_close_short": 0.00,
    "open_frac": 1,
    "close_frac": 1,
}

HYPER = {
    "max_freshness_ms": None,
    "max_latency_okx_ms": None,  # заполним p95 после load
    "max_latency_bybit_ms": None,
    "avg_window_sec": 10.0,
    "Trade_Lat": 100,  # мс; fill на тике той же монеты
    "Check_volume": False,
    "position_size": 10.0,
    "position_frac": 1.0,
    "fee_rate": 0.00075,  # blend гира 1.0; taker-taker — позже
    "reject_fill_across_gap": False,  # True → не fill через скачок ≫ Trade_Lat
    "gap_fill_slack_ms": 1000,
}

print("LEAN_TICKS", LEAN_TICKS, "exists", LEAN_TICKS.exists())
print("window", START, "→", END, "(END exclusive)")
print("MAX_COINS", MAX_COINS, "K", K, "allowlist", COIN_ALLOWLIST)
print(
    "DAY_MODE", DAY_MODE, "CHUNK_HOURS", CHUNK_HOURS,
    "PRUNE_LEAN_COLS", PRUNE_LEAN_COLS, "RUN_ARM_B", RUN_ARM_B,
)
print("USE_REGIME_TOPN", USE_REGIME_TOPN, "REGIME_TOP_N", REGIME_TOP_N,
      "bars", REGIME_BAR_ROOT)
print("VARIATION", VARIATION)
print("HYPER", {k: v for k, v in HYPER.items() if k not in ("max_latency_okx_ms", "max_latency_bybit_ms")})

LEAN_TICKS /Users/mishatrubik/Desktop/spread/output/lean_ticks exists True
window 2026-08-17T00:00:00Z → 2026-08-18T02:00:00Z (END exclusive)
MAX_COINS 15 K 1 allowlist ['BTC', 'ETH']
DAY_MODE True CHUNK_HOURS 2 PRUNE_LEAN_COLS True RUN_ARM_B True
USE_REGIME_TOPN False REGIME_TOP_N 10 bars /Users/mishatrubik/Desktop/spread/output/okx_bar5m_hist_regime
VARIATION {'thresh_open_long': 0.02, 'thresh_open_short': 0.02, 'thresh_close_long': 0.02, 'thresh_close_short': 0.02, 'open_frac': 1, 'close_frac': 1}
HYPER {'max_freshness_ms': None, 'avg_window_sec': 10.0, 'Trade_Lat': 100, 'Check_volume': False, 'position_size': 10.0, 'position_frac': 1.0, 'fee_rate': 0.00075, 'reject_fill_across_gap': False, 'gap_fill_slack_ms': 1000}


## Загрузка тиков (lean L1) + спреды при чтении


In [44]:
start_ms = parse_ts_ms(START)
end_ms = parse_ts_ms(END)
if end_ms <= start_ms:
    raise ValueError("END must be after START")

from research.lean_ticks_io import gear2_lean_columns

# Resolve coin universe before Arrow load (pass coins=; do not load U then filter).
if COIN_ALLOWLIST:
    coins = sorted({c.upper() for c in COIN_ALLOWLIST})
else:
    import pandas as _pd
    uni = _pd.read_csv(REPO / "bybit_okx_universe.csv")["base_coin"].astype(str).str.upper()
    coins = sorted(c for c in uni.unique().tolist() if is_crypto(c))
if MAX_COINS and MAX_COINS > 0:
    coins = coins[: int(MAX_COINS)]
coins_set = set(coins)

read_cols = (
    gear2_lean_columns(check_volume=bool(HYPER.get("Check_volume", False)))
    if PRUNE_LEAN_COLS
    else None
)

if DAY_MODE:
    # p95 from first chunk only (same rule as slice p95, scoped to chunk 0).
    chunk_ms = int(CHUNK_HOURS * 3600 * 1000)
    p95_end = min(end_ms, start_ms + chunk_ms)
    t0 = time.perf_counter()
    df_p95, files = read_and_prepare_lean_ticks(
        LEAN_TICKS,
        start_ms,
        p95_end,
        coins=coins_set,
        columns=read_cols,
        slim_backtest=True,
        check_volume=bool(HYPER.get("Check_volume", False)),
        need_freshness=HYPER.get("max_freshness_ms") is not None,
    )
    print(f"DAY_MODE p95 sample in {time.perf_counter()-t0:.1f}s  files={len(files)} ticks={len(df_p95)}")
    p95_okx = float(df_p95["okx_latency_ms"].quantile(0.95))
    p95_bybit = float(df_p95["bybit_latency_ms"].quantile(0.95))
    del df_p95
    df = None  # filled by chunked runner
else:
    t0 = time.perf_counter()
    df, files = read_and_prepare_lean_ticks(
        LEAN_TICKS,
        start_ms,
        end_ms,
        coins=coins_set,
        columns=read_cols,
        slim_backtest=bool(PRUNE_LEAN_COLS),
        check_volume=bool(HYPER.get("Check_volume", False)),
        need_freshness=HYPER.get("max_freshness_ms") is not None,
    )
    print(f"load+prepare in {time.perf_counter()-t0:.1f}s  files={len(files)}")
    df = df.sort_values(["event_local_ts_ms", "base_coin"], kind="mergesort").reset_index(
        drop=True
    )
    p95_okx = float(df["okx_latency_ms"].quantile(0.95))
    p95_bybit = float(df["bybit_latency_ms"].quantile(0.95))

if not np.isfinite(p95_okx) or not np.isfinite(p95_bybit):
    raise ValueError(f"non-finite pooled p95 okx={p95_okx!r} bybit={p95_bybit!r}")
HYPER["max_latency_okx_ms"] = p95_okx
HYPER["max_latency_bybit_ms"] = p95_bybit

print(
    f"coins={len(coins)} crypto_filter "
    f"{'allowlist' if COIN_ALLOWLIST else 'is_crypto'} | "
    f"DAY_MODE={DAY_MODE} prune={PRUNE_LEAN_COLS}"
)
print(f"coins {coins[:20]}{'...' if len(coins)>20 else ''}")
print(f"HYPER p95_okx={p95_okx:.3f} p95_bybit={p95_bybit:.3f} (pooled on {'first chunk' if DAY_MODE else 'this slice'})")
if df is not None:
    print(f"ticks={len(df)} | {df['event_dt'].min()} → {df['event_dt'].max()}")


read 21 files, raw rows=274454
DAY_MODE p95 sample in 4.8s  files=21 ticks=274454
coins=2 crypto_filter allowlist | DAY_MODE=True prune=True
coins ['BTC', 'ETH']
HYPER p95_okx=35.000 p95_bybit=24.000 (pooled on first chunk)


## Gear 1.5 arm B — Top‑10 только на вход

Канон закрытого гира 1.5: soft short blend `α≈0.75`, composite `geom`, бары OKX `output/okx_bar5m_hist_regime/`, `MA_LONG=288` как heatmap. N не подбирается по PnL (`REGIME_TOP_N = 10`).

Causal: score бара `[t−5m, t)` только для тиков `≥ t`. На close не применяется. Нет бара у монеты (QNT, USDC) → не в top‑N, open блокируется (`n_filtered_not_topn`). `K=1` elif 1.0 без изменений. Флаг выкл. → эталон без кластера.


In [45]:
REGIME_TOPN_BY_BAR = None
REGIME_FRAMES = None
print(describe_canon())
t_reg = time.perf_counter()
REGIME_FRAMES, _root, _no_feat = load_crypto_feature_frames(
    start_ms=start_ms,
    end_ms=end_ms,
    root=REGIME_BAR_ROOT,
    params=canon_ma_params(),
    workers=int(REGIME_TOPN_WORKERS),
)
REGIME_TOPN_BY_BAR = build_topn_by_bar(
    REGIME_FRAMES,
    start_ms=start_ms,
    end_ms=end_ms,
    top_n=int(REGIME_TOP_N),
    params=canon_ma_params(),
)
_tick_no_hist = no_hist_tick_coins(coins, REGIME_FRAMES.keys())
print(
    f"1.5 top-{REGIME_TOP_N}: frames={len(REGIME_FRAMES)} bars_with_sets="
    f"{len(REGIME_TOPN_BY_BAR)} in {time.perf_counter()-t_reg:.1f}s  root={_root}"
)
print("preview top-N at START", preview_topn_at(REGIME_TOPN_BY_BAR, start_ms))
if _tick_no_hist:
    print("tick coins with no hist bars (fail-closed on open):", _tick_no_hist)
if _no_feat:
    print(f"crypto hist coins without features: {len(_no_feat)} (not in ranking)")
if not REGIME_TOPN_BY_BAR:
    raise RuntimeError("Top-N map is empty — check hist bars (needed for arm B)")
print(
    "USE_REGIME_TOPN", USE_REGIME_TOPN,
    "— main run is arm A if False; arm B uses this map in the close-out table",
)


1.5 arm B Top-N: blend α=0.75 short=6 long=288 variant=geom  (bar [t−5m, t) for ticks ≥ t)
1.5 top-10: frames=240 bars_with_sets=312 in 12.5s  root=/Users/mishatrubik/Desktop/spread/output/okx_bar5m_hist_regime
preview top-N at START ['CBRS', 'DATA', 'DYDX', 'GPS', 'IRYS', 'MON', 'RENDER', 'SHAZ', 'SOPH', 'THETA']
USE_REGIME_TOPN False — main run is arm A if False; arm B uses this map in the close-out table


## Движок: копия elif гира 1.0 + слот K=1

Fill ищется **только вперёд по тикам той же монеты** (не любой тик рынка).
Пока слот занят (`pos`) — open по другим монетам: `n_filtered_slot_busy`.
Пока `pending` (ждём fill) — open-порог по **чужим** монетам: `n_filtered_pending_skip` (не смешивать со `slot_busy`, эталон v0 по `slot_busy` не меняется).

При `regime_topn` (arm B): на **open** монета должна быть в Top‑`REGIME_TOP_N` канона 1.5 (blend α≈0.75 geom). На close фильтр не ставится. Нет hist-баров (QNT, USDC) → fail closed, `n_filtered_not_topn`.


In [46]:
from research.gear2_backtest import (
    BacktestResult,
    assert_k1_invariants,
    closeout_row,
    run_backtest_chunked,
    run_backtest_market,
)

print(
    "engine research.gear2_backtest | K=1 | pending_skip explicit | "
    f"DAY_MODE={DAY_MODE} chunked={'run_backtest_chunked' if DAY_MODE else 'run_backtest_market'}"
)


engine research.gear2_backtest | K=1 | pending_skip explicit | DAY_MODE=True chunked=run_backtest_chunked


## Прогон


In [47]:
t0 = time.perf_counter()
if DAY_MODE:
    result = run_backtest_chunked(
        LEAN_TICKS,
        start_ms,
        end_ms,
        variation=VARIATION,
        hyper=HYPER,
        k=K,
        regime_topn=REGIME_TOPN_BY_BAR if USE_REGIME_TOPN else None,
        coins=coins_set,
        chunk_ms=int(CHUNK_HOURS * 3600 * 1000),
    )
    print(f"run_backtest_chunked in {time.perf_counter()-t0:.1f}s")
else:
    result = run_backtest_market(
        df,
        variation=VARIATION,
        hyper=HYPER,
        k=K,
        regime_topn=REGIME_TOPN_BY_BAR if USE_REGIME_TOPN else None,
    )
    print(f"run_backtest_market in {time.perf_counter()-t0:.1f}s")
n_open = 1 if result.open_position is not None else 0
n_closed = sum(1 for t in result.trades if t.status == "closed")
print(
    f"ticks={result.n_ticks} coins={result.n_coins} | "
    f"closed={n_closed} open_at_end={n_open} metric={result.metric:.6f} "
    f"fees_total={result.fees_total:.6f}"
)
print(
    f"signals raw={result.n_signals_raw} passed={result.n_signals_passed} | "
    f"filt freshness={result.n_filtered_by_freshness} latency={result.n_filtered_by_latency} "
    f"avg={result.n_filtered_by_avg} size={result.n_filtered_by_size} "
    f"slot_busy={result.n_filtered_slot_busy} pending_skip={result.n_filtered_pending_skip} "
    f"not_topn={result.n_filtered_not_topn} pending_missed={result.n_pending_missed}"
)


read 21 files, raw rows=274454
chunk 0: [1786924800000,1786932000000) ticks=274454 closed+=0 pending=n open=Y
read 21 files, raw rows=269181
chunk 1: [1786932000000,1786939200000) ticks=268884 closed+=0 pending=n open=Y
read 21 files, raw rows=231377
chunk 2: [1786939200000,1786946400000) ticks=231002 closed+=1 pending=n open=Y
read 20 files, raw rows=272445
chunk 3: [1786946400000,1786953600000) ticks=272125 closed+=0 pending=n open=Y
read 21 files, raw rows=265503
chunk 4: [1786953600000,1786960800000) ticks=264955 closed+=0 pending=n open=Y
read 21 files, raw rows=268349
chunk 5: [1786960800000,1786968000000) ticks=267941 closed+=0 pending=n open=Y
read 21 files, raw rows=315608
chunk 6: [1786968000000,1786975200000) ticks=315211 closed+=0 pending=n open=Y
read 20 files, raw rows=343748
chunk 7: [1786975200000,1786982400000) ticks=343165 closed+=0 pending=n open=Y
read 22 files, raw rows=332631
chunk 8: [1786982400000,1786989600000) ticks=332053 closed+=0 pending=n open=Y
read 21 fi

## Отчёт и инварианты K=1

`metric` — сумма PnL **закрытых** сделок (как гир 1.0). Открытая на конце позиция в метрику не входит.

Таблица сделок ниже — журнал **сигнала и исполнения** (`Trade_Lat`): время/спред на тике сигнала и на тике fill. Для `open_at_end` выхода нет (нет close fill).

Таблица A vs B — **отсевы и сигналы**, не выбор победителя по прибыли.

In [48]:
OPEN_AT_END = "open-at-end"
NO_CLOSE_FILL = "no close fill"


def _utc_ms(ts):
    if ts is None:
        return None
    t = pd.Timestamp(float(ts), unit="ms", tz="UTC")
    return t.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + " UTC"


def _wait_ms(signal_ts, fill_ts):
    if signal_ts is None or fill_ts is None:
        return None
    return float(fill_ts) - float(signal_ts)


def trades_frame(res: BacktestResult) -> pd.DataFrame:
    """Журнал сделок: сигнал vs fill. Поля уже на Trade; новых полей движка нет."""
    rows = []
    for t in res.trades:
        closed = t.status == "closed"
        sig_open_ts = t.signal_open_ts if t.signal_open_ts is not None else t.open_ts
        rows.append(
            {
                "coin": t.base_coin,
                "side": t.side,
                "status": "closed" if closed else "open_at_end",
                "entry_signal_utc": _utc_ms(sig_open_ts),
                "entry_fill_utc": _utc_ms(t.open_ts),
                "exit_signal_utc": _utc_ms(t.signal_close_ts) if closed else OPEN_AT_END,
                "exit_fill_utc": _utc_ms(t.close_ts) if closed else NO_CLOSE_FILL,
                "spread_signal_open": t.signal_open_price,
                "spread_fill_open": t.open_price,
                "spread_signal_close": t.signal_close_price if closed else None,
                "spread_fill_close": t.close_price if closed else None,
                "wait_open_ms": _wait_ms(sig_open_ts, t.open_ts),
                "wait_close_ms": _wait_ms(t.signal_close_ts, t.close_ts) if closed else None,
            }
        )
    return pd.DataFrame(rows)


n_open_tf = 1 if result.open_position is not None else 0
n_closed_tf = sum(1 for t in result.trades if t.status == "closed")
tf = trades_frame(result)
print(
    "журнал сделок (сигнал ≠ fill при Trade_Lat>0) | "
    f"closed={n_closed_tf} open_at_end={n_open_tf} rows={len(tf)}"
)
if n_closed_tf == 0:
    print("closed=0 в текущем CONFIG (debug-окно); ниже open_at_end и/или пустая таблица, не канон 4h")
if tf.empty:
    print("no trades in this window (narrow CONFIG or quiet tape)")
else:
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 36)
    display(tf)

# --- K=1 invariants ---
assert_k1_invariants(result)
print("K=1 invariants OK | overlapping_positions=0")

# synthetic two-coin slot test (does not use parquet)
def _synth_row(ts, coin, sl, ss):
    return {
        "event_local_ts_ms": ts,
        "base_coin": coin,
        "spread_long": sl,
        "spread_short": ss,
        "okx_latency_ms": 1.0,
        "bybit_latency_ms": 1.0,
        "event_dt": pd.Timestamp(ts, unit="ms", tz="UTC"),
        "okx_bid_size": 100.0,
        "okx_ask_size": 100.0,
        "bybit_bid_size": 100.0,
        "bybit_ask_size": 100.0,
    }


synth = pd.DataFrame(
    [
        _synth_row(0, "AAA", 2.0, 0.0),
        _synth_row(50, "BBB", 2.0, 0.0),
        _synth_row(200, "AAA", 2.0, 0.0),  # fill AAA open (Trade_Lat=100)
        _synth_row(250, "BBB", 2.0, 0.0),  # slot busy
        _synth_row(400, "AAA", 0.0, 2.0),  # close long on AAA
        _synth_row(550, "AAA", 0.0, 2.0),  # fill close
    ]
)
hyper_loose = {
    **HYPER,
    "max_latency_okx_ms": 10_000.0,
    "max_latency_bybit_ms": 10_000.0,
    "avg_window_sec": None,
    "Trade_Lat": 100,
    "Check_volume": False,
    "fee_rate": 0.0,
}
syn_res = run_backtest_market(synth, variation=VARIATION, hyper=hyper_loose, k=1)
syn_closed = [t for t in syn_res.trades if t.status == "closed"]
assert len(syn_closed) == 1, syn_res
assert syn_closed[0].base_coin == "AAA"
assert syn_res.n_filtered_slot_busy >= 1
assert syn_res.n_filtered_pending_skip >= 1
print(
    "synthetic two-coin: closed AAA only, "
    f"slot_busy={syn_res.n_filtered_slot_busy} pending_skip={syn_res.n_filtered_pending_skip} OK"
)

# 1.5 arm B: open blocked when coin is not in Top-N (synthetic; not parquet)
_b = completed_bar_start_ms(0)
_topn_aaa = {_b: frozenset({"AAA"}), completed_bar_start_ms(200): frozenset({"AAA"})}
synth_bbb = pd.DataFrame(
    [
        _synth_row(0, "BBB", 2.0, 0.0),
        _synth_row(200, "BBB", 2.0, 0.0),
    ]
)
syn_block = run_backtest_market(
    synth_bbb, variation=VARIATION, hyper=hyper_loose, k=1, regime_topn=_topn_aaa
)
assert syn_block.n_filtered_not_topn >= 1, syn_block
assert not [t for t in syn_block.trades if t.status == "closed"]
assert syn_block.open_position is None
syn_off = run_backtest_market(
    synth_bbb, variation=VARIATION, hyper=hyper_loose, k=1, regime_topn=None
)
assert syn_off.open_position is not None or any(t.status == "closed" for t in syn_off.trades)
print(
    f"synthetic not-topn: blocked BBB n_filtered_not_topn={syn_block.n_filtered_not_topn}; "
    f"flag off opens={syn_off.open_position is not None} OK"
)
print(f"live run n_filtered_not_topn={result.n_filtered_not_topn}")

# --- Stage 3 A vs B: filters and signal counts, not PnL ---
if DAY_MODE or not RUN_ARM_B:
    print(
        "skip dual A+B close-out "
        f"(DAY_MODE={DAY_MODE}, RUN_ARM_B={RUN_ARM_B}); arm A result above is the day ledger"
    )
elif df is None:
    print("skip A+B: df is None")
else:
    assert REGIME_TOPN_BY_BAR, "Top-N map required for arm B"
    t_ab = time.perf_counter()
    res_a = run_backtest_market(df, variation=VARIATION, hyper=HYPER, k=K, regime_topn=None)
    res_b = run_backtest_market(
        df, variation=VARIATION, hyper=HYPER, k=K, regime_topn=REGIME_TOPN_BY_BAR
    )
    print(f"A+B close-out runs in {time.perf_counter()-t_ab:.1f}s")
    assert_k1_invariants(res_a)
    assert_k1_invariants(res_b)
    assert res_a.n_filtered_not_topn == 0, "arm A must not use Top-N"
    closeout = pd.DataFrame([closeout_row("A_flag_off", res_a), closeout_row("B_topn_open", res_b)])
    display(closeout)
    print("A vs B: compare raw/passed/slot_busy/pending_skip/gates/not_topn. Do not pick a winner by metric.")
    print("C (regime_on) and D (random entry) are not in this notebook — Gear 2.2.")

журнал сделок (сигнал ≠ fill при Trade_Lat>0) | closed=1 open_at_end=1 rows=2


,coin,side,status,entry_signal_utc,entry_fill_utc,exit_signal_utc,exit_fill_utc,spread_signal_open,spread_fill_open,spread_signal_close,spread_fill_close,wait_open_ms,wait_close_ms
0,BTC,short,closed,2026-08-17 00:05:15.597 UTC,2026-08-17 00:05:15.727 UTC,2026-08-17 05:11:03.237 UTC,2026-08-17 05:11:03.338 UTC,0.023534,0.023534,0.020329,0.020171,130.0,101.0
1,BTC,long,open_at_end,2026-08-17 05:11:03.339 UTC,2026-08-17 05:11:03.458 UTC,open-at-end,no close fill,0.020171,0.020171,NaN,NaN,119.0,NaN


K=1 invariants OK | overlapping_positions=0
synthetic two-coin: closed AAA only, slot_busy=1 pending_skip=1 OK
synthetic not-topn: blocked BBB n_filtered_not_topn=2; flag off opens=True OK
live run n_filtered_not_topn=0
skip dual A+B close-out (DAY_MODE=True, RUN_ARM_B=True); arm A result above is the day ledger


## Графики убраны

Ряды спреда, гистограммы и counterfactual-окна больше не рисуются. Журнал сделок — таблица в ячейке «Отчёт».

In [49]:
print("графики сделок убраны — см. таблицу журнала выше")

графики сделок убраны — см. таблицу журнала выше


## Обзор тиков одной монеты — убран

Отдельный BTC-график 5–19 августа больше не строится в этом ноутбуке. HTML `output/gear2_coin_overview/` на диске не трогаем и не расширяем.

In [50]:
print("обзорный график монеты убран")

обзорный график монеты убран


## Этапы закрытия гира 2

Канон: [`docs/strategy-gears.md`](docs/strategy-gears.md). Снимок счётчиков: [`docs/gear-2-close-20260825.md`](docs/gear-2-close-20260825.md).

**Статус:** гир 2 = закрыт (контур; 2.2 вне scope). Validator 2026-08-25 **YELLOW** (таблица 1:1). Critic accept-with-caveats. Этот ноутбук **не** воспроизводит 4h канон: debug `CONFIG` + зачищенные outputs. Выход ячеек — журнал сделок, не графики.

| Этап | Суть | Здесь |
|------|------|--------|
| **0** | Replay crypto, elif 1.0, `K=1` | сделано |
| **1** | Lean-тики, fail-closed, `HYPER` 1.0 записан (не ретьюнить) | сделано |
| **2** | `n_filtered_pending_skip`; `K=1` без двух позиций; `K>1` не требуется | сделано в коде |
| **3** | **A** флаг выкл.; **B** Top‑N на open. C/D → гир 2.2 | канон A/B в docs, не cells |
| **4** | Таблица отсевов + инварианты + Validator | канон в `gear-2-close-20260825.md` |

Дальше гир **2.5** (политика размера). Вход = доступный флаг кластера, не «кластер — вселенная». Поиск `VARIATION` — гир 3.